# 10年定着予測 - GBDT Ensemble with EDA-Driven Feature Engineering (v2)

**目的**: `11_ensemble_gbdt_advanced_features.ipynb`（Public Score: 0.606518, Stacking提出）を、
EDAレポート（`data_exploration_report.md` / `data_exploration_v2_report.md`）と
提出結果レポート（`submit_result_report.md`）で明らかになった知見に基づいて改善する。

## 11_からの主な改善点

### A. 未活用だった強力な特徴量の追加
1. **部署IDのTarget Encoding（KFold + スムージング）** — 11_では`初期部署ID`は完全にドロップされ未使用だった。
   EDA v2で「部署IDはn≥10のグループで定着率0%〜82%とレンジが極めて大きい強力な特徴量候補」と判明したため、
   CVベースのスムージング付きTarget Encodingとして新規に活用する（5節）。
2. **欠勤日数の時系列パターン特徴量** — EDAレポート18.2「退職前3ヶ月で欠勤日数が42.9%急増」という最重要シグナルに対応し、
   後半/前半比・連続欠勤フラグ・最長連続欠勤月数を追加。
3. **職種別の乖離特徴量** — EDA「研修時間と定着率の負の相関は職種構造に起因する見かけの相関」という知見を踏まえ、
   絶対値ではなく職種内偏差・比率で特徴量を文脈化。
4. **入社四半期×入社区分の交互作用（is_Q2_新卒）** — 「Q2入社の定着率の低さは新卒一括採用比率の違いが主因」という知見を直接特徴量化。
5. **360度評価タイミング特徴量** — 「3-4ヶ月で評価開始した群の定着率が最高、標準の5-6ヶ月が最低」という知見を特徴量化。
6. **給与の相対化特徴量** — EDA v2「初任給・月例給与にTrain/Test分布シフトあり（KS統計量0.17〜0.20）」を踏まえ、
   絶対額に加え等級内偏差・入社区分内偏差を追加し、分布シフトに頑健にする。
7. **上司チーム規模特徴量** — 「上司IDは1人あたり平均2名でTarget Encoding不可能」という知見を踏まえ、
   上司ID自体は使わず「初期上司の同時受持ち人数」という代替特徴量を追加。

### B. 当初計画したが11_で未実装だった特徴量の実装
11_の冒頭Markdownには「多項式特徴量」「ビン化特徴量」「複雑な交互作用」「時系列ラグ特徴量」「比率・割合特徴量」が
計画として記載されていたが、実際のコードには存在しなかった（`submit_result_report.md`で指摘済み）。
本ノートブックではこれらを実装する。

### C. モデリングの改善
1. **TimeSeriesSplitの分割数を5→6に増加** — 11_ではOOF LogLoss（0.585）とPublic Score（0.607）の間に
   約0.02のギャップがあり、fold数の少なさによるOOFの楽観性が一因と考えられるため。
2. **XGBoostのOptuna試行回数を10→20に増加** — 11_ではXGBoostが3モデル中最も弱く（OOF 0.611 vs 他0.587程度）、
   Stackingでもほぼ無視されていたため、探索を強化する。
3. **Optimized Weighted Average（新規のアンサンブル手法）** — 11_の重み付け平均は「スコアの逆数」というヒューリスティックだったが、
   OOF Log Lossを直接最小化する重みを`scipy.optimize`で求める方式を追加する。
4. **Stackingメタモデルの正則化強度をCVで選択** — EDA v2で指摘された年齢・前職経験・初任給の多重共線性（VIF 7〜8台）を踏まえ、
   固定の`LogisticRegression(C=1.0)`ではなく`LogisticRegressionCV`でL2正則化強度を選択する。

## 実行環境
`11_`と同様、Google Colab（GPU: T4等）での実行を想定。Google Driveのマウントが必要。


In [1]:
# catboostとoptunaのインストール
!pip install catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 15.5 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 27.6 MB/s eta 0:00:00


In [2]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sat Aug  8 13:59:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import psutil

# メモリ情報を取得
mem = psutil.virtual_memory()
total_memory_gb = mem.total / (1024**3)
available_memory_gb = mem.available / (1024**3)
used_memory_gb = mem.used / (1024**3)

print(f"総メモリ: {total_memory_gb:.2f} GB")
print(f"利用可能メモリ: {available_memory_gb:.2f} GB")
print(f"使用済みメモリ: {used_memory_gb:.2f} GB")

import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# プロジェクトルートの設定（Google Drive）
PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Mounted at /content/drive


In [4]:
import datetime
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import KFold

# モジュールのインポート（v2: TimeSeriesSplit対応版）
from common.lgbm.lgbm_model_v2 import run_lgb
from common.xgboost.xgb_model_v2 import run_xgb
from common.catboost.cat_model_v2 import run_cat
from common.lgbm.lgbm_model_optuna_v2 import run_lgb_optuna
from common.xgboost.xgb_model_optuna_v2 import run_xgb_optuna
from common.catboost.cat_model_optuna_v2 import run_cat_optuna
from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")

# 乱数シードの固定
SEED = 42
seed_everything(seed=SEED)

# ターゲット列とID列の設定
TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

# 表示設定
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
# スクリプト名・日付・保存パスの設定
SCRIPT_NAME = "12_ensemble_gbdt_eda_driven_features"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

# ログディレクトリ
LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

# 出力ディレクトリ
OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 保存パス
SUBMISSION_SIMPLE_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_simple_avg.csv"
SUBMISSION_WEIGHTED_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_weighted_avg.csv"
SUBMISSION_OPT_WEIGHTED_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_opt_weighted_avg.csv"
SUBMISSION_STACKING_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_stacking.csv"

# モデル保存ディレクトリ
SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Saved Models Directory: {SAVED_MODELS_DIR}")

[2026-08-08 14:00:52] [INFO] === [12_ensemble_gbdt_eda_driven_features] 実験開始 ===


INFO:12_ensemble_gbdt_eda_driven_features:=== [12_ensemble_gbdt_eda_driven_features] 実験開始 ===


[2026-08-08 14:00:53] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260808


INFO:12_ensemble_gbdt_eda_driven_features:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260808


[2026-08-08 14:00:53] [INFO] Saved Models Directory: /content/drive/MyDrive/jaggle_2026/saved_models/20260808/12_ensemble_gbdt_eda_driven_features


INFO:12_ensemble_gbdt_eda_driven_features:Saved Models Directory: /content/drive/MyDrive/jaggle_2026/saved_models/20260808/12_ensemble_gbdt_eda_driven_features


In [6]:
# データの読み込み
INPUT_DIR = PROJECT_ROOT / "data" / "input"

# 属性データの読み込み (社員1名 = 1行)
train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")

# 月次データの読み込み (社員1名 × 24か月 = 複数行)
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

# ターゲット変数の取得
y_train = train_persona[TARGET_COL]

# 社員IDリスト
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-08 14:00:59] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:12_ensemble_gbdt_eda_driven_features:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-08 14:00:59] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:12_ensemble_gbdt_eda_driven_features:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-08 14:00:59] [INFO] 定着率: 0.5647


INFO:12_ensemble_gbdt_eda_driven_features:定着率: 0.5647


[2026-08-08 14:00:59] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:12_ensemble_gbdt_eda_driven_features:Train IDs: 2761, Test IDs: 2502


## 1. 月次データの集約特徴量（11_から継承・変更なし）

0-23ヶ月の月次データから、統計量ベース・時期別統計量・トレンド特徴量を生成する。
このロジックは11_で有効に機能していたため変更しない。

In [7]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """
    月次データから集約特徴量を生成
    """
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            # 統計量
            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            # 時期別
            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            # 後半/前半比（欠勤日数などの急増シグナルを捉える。EDAレポート18.2「追加1」対応）
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            # トレンド
            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)

print("✅ 月次集約特徴量関数定義完了（後半/前半比を追加）")

✅ 月次集約特徴量関数定義完了（後半/前半比を追加）


In [8]:
def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)

def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)

def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)

print("✅ 月次カテゴリ変化+欠損値+ドメイン特徴量関数定義完了")

✅ 月次カテゴリ変化+欠損値+ドメイン特徴量関数定義完了


## 2. 高度な統計特徴量・クラスター特徴量（11_から継承・変更なし）

In [9]:
def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)

def create_cluster_features(monthly_df, employee_ids, n_clusters=5):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]

print("✅ 高度な統計特徴量・クラスター特徴量関数定義完了")

✅ 高度な統計特徴量・クラスター特徴量関数定義完了


## 3. EDA駆動の新規特徴量（本ノートブックでの新規追加）

`data_exploration_v2_report.md` および `data_exploration_report.md`（18.2節）の知見をもとに、
11_で未活用だった特徴量を追加する。

- **部署IDのTarget Encoding**（v2レポート5節）: 初期部署IDは11_で完全にドロップされていたが、
  n≥10のグループで定着率が0%〜82%と非常に大きくばらつく強力な特徴量候補。KFold内で計算しスムージングを施すことでリークと過学習を防ぐ。
- **欠勤日数パターン特徴量**（初回レポート18.2「追加1・最優先」）: 退職前の欠勤急増シグナルを、後半/前半比・連続欠勤・最長連続欠勤月数として捕捉。
- **360度評価タイミング特徴量**（初回レポート18.2「追加4」/ 17.5節）: 初回評価月と定着率の関係（3-4ヶ月がベスト、5-6ヶ月が最悪）を特徴量化。
- **時系列ボラティリティ（ラグ）特徴量**: 11_計画にあった「時系列ラグ特徴量」の実装。月次の変動の激しさ（前月差分の平均絶対値）を捕捉。
- **比率・割合特徴量**: 11_計画にあった「比率・割合特徴量」の実装。有給取得率、360度評価の項目間ばらつきを追加。
- **上司チーム規模特徴量**（v2レポート5節）: 上司IDは単独では実用困難（n≥10のグループが0）なため、
  上司ID自体は使わず「初期上司が同時に受け持つ人数」という代替特徴量を追加。

In [10]:
def create_department_target_encoding(train_persona, test_persona, y_train, seed=42, n_splits=5, smoothing=10):
    """
    初期部署IDのKFold + スムージング付きTarget Encoding

    EDA v2レポート5節: 部署IDはn≥10のグループで定着率0.0〜0.818とレンジが極めて大きく、
    強力な特徴量候補である一方、461グループ中409グループがn<10のため、
    ベイズ的スムージング（グループサイズが小さいほど全体平均に縮約）が必須。
    Trainに対してはKFold外の予測を使うことでリークを防ぐ。
    """
    col = "初期部署ID"
    global_mean = y_train.mean()

    train_te = np.zeros(len(train_persona))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    dept_train = train_persona[col].values
    y_arr = y_train.values

    for tr_idx, val_idx in kf.split(train_persona):
        df_tr = pd.DataFrame({col: dept_train[tr_idx], "y": y_arr[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        train_te[val_idx] = pd.Series(dept_train[val_idx]).map(mapping).fillna(global_mean).values

    # Test用: 全Trainデータで計算したスムージング済みマッピングを適用
    df_full = pd.DataFrame({col: dept_train, "y": y_arr})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        "社員ID": train_persona["社員ID"].values,
        "dept_target_enc": train_te,
        "dept_size": train_persona[col].map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        "社員ID": test_persona["社員ID"].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def create_eda_driven_features(monthly_df, employee_ids):
    """
    欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量をまとめて生成
    （1回のループで複数の特徴量グループを計算し、実行時間を抑える）
    """
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        # --- 欠勤日数パターン（初回レポート18.2「追加1」） ---
        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        # --- 360度評価タイミング（初回レポート18.2「追加4」/ 17.5節） ---
        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        # --- 時系列ボラティリティ（ラグ特徴量、11_計画の未実装分） ---
        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        # --- 比率・割合特徴量（11_計画の未実装分） ---
        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """
    初期（経過月数=0）時点で、同じ上司IDを持つ社員数（ベクトル化・高速）

    v2レポート5節: 上司IDは1人あたり平均2名しか受け持っておらずTarget Encodingは不可能なため、
    上司ID自体の代わりに「チーム規模」という代替特徴量を使う。
    """
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ EDA駆動の新規特徴量関数定義完了")

✅ EDA駆動の新規特徴量関数定義完了


## 4. 特徴量生成の実行

定義した関数を使用して、Train/Testデータの特徴量を生成する。

In [11]:
logger.info("-" * 60)
logger.info("特徴量生成開始")
logger.info("-" * 60)

logger.info("月次集約特徴量を生成中...")
train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)
logger.info(f"Train: {train_monthly_agg.shape}, Test: {test_monthly_agg.shape}")

logger.info("月次カテゴリ変化特徴量を生成中...")
train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)
logger.info(f"Train: {train_cat_change.shape}, Test: {test_cat_change.shape}")

logger.info("欠損値特徴量を生成中...")
train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)
logger.info(f"Train: {train_missing.shape}, Test: {test_missing.shape}")

logger.info("ドメイン知識特徴量を生成中...")
train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)
logger.info(f"Train: {train_domain.shape}, Test: {test_domain.shape}")

logger.info("高度な統計特徴量を生成中...")
train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)
logger.info(f"Train: {train_advanced_stats.shape}, Test: {test_advanced_stats.shape}")

logger.info("クラスター特徴量を生成中...")
train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5)
logger.info(f"Train: {train_cluster.shape}, Test: {test_cluster.shape}")

logger.info("[NEW] 部署ID Target Encodingを生成中...")
train_dept_te, test_dept_te = create_department_target_encoding(
    train_persona, test_persona, y_train, seed=SEED, n_splits=5, smoothing=10
)
logger.info(f"Train: {train_dept_te.shape}, Test: {test_dept_te.shape}")

logger.info("[NEW] EDA駆動特徴量（欠勤パターン・評価タイミング・ボラティリティ・比率）を生成中...")
train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)
logger.info(f"Train: {train_eda_feats.shape}, Test: {test_eda_feats.shape}")

logger.info("[NEW] 上司チーム規模特徴量を生成中...")
train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)
logger.info(f"Train: {train_mgr.shape}, Test: {test_mgr.shape}")

[2026-08-08 14:00:59] [INFO] ------------------------------------------------------------


INFO:12_ensemble_gbdt_eda_driven_features:------------------------------------------------------------


[2026-08-08 14:00:59] [INFO] 特徴量生成開始


INFO:12_ensemble_gbdt_eda_driven_features:特徴量生成開始


[2026-08-08 14:00:59] [INFO] ------------------------------------------------------------


...

INFO:12_ensemble_gbdt_eda_driven_features:Train: (2761, 12), Test: (2502, 12)


[2026-08-08 14:07:46] [INFO] [NEW] 上司チーム規模特徴量を生成中...


INFO:12_ensemble_gbdt_eda_driven_features:[NEW] 上司チーム規模特徴量を生成中...


[2026-08-08 14:07:46] [INFO] Train: (2761, 2), Test: (2502, 2)


INFO:12_ensemble_gbdt_eda_driven_features:Train: (2761, 2), Test: (2502, 2)


In [12]:
logger.info("テキスト特徴量を生成中...")
text_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]
train_persona["text_total_chars"] = train_persona[text_cols].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[text_cols].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
for col in text_cols:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)

logger.info("時間的特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])
train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

logger.info("交互作用特徴量を生成中...")
train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

# [NEW] 入社四半期 × 入社区分（初回レポート18.2「追加3」/ 17.4節）
# Q2（4-6月）入社の定着率の低さは「新卒一括採用比率の違い」が主因という知見を直接特徴量化
train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

# [NEW] 複雑な交互作用特徴量（11_計画の未実装分）: 入社区分 × 初期職種
train_persona["combo_job_category"] = train_persona["入社区分"].astype(str) + "_" + train_persona["初期職種"].astype(str)
test_persona["combo_job_category"] = test_persona["入社区分"].astype(str) + "_" + test_persona["初期職種"].astype(str)

logger.info("特徴量処理完了")

[2026-08-08 14:07:46] [INFO] テキスト特徴量を生成中...


INFO:12_ensemble_gbdt_eda_driven_features:テキスト特徴量を生成中...


[2026-08-08 14:07:46] [INFO] 時間的特徴量を生成中...


INFO:12_ensemble_gbdt_eda_driven_features:時間的特徴量を生成中...


[2026-08-08 14:07:46] [INFO] 交互作用特徴量を生成中...


INFO:12_ensemble_gbdt_eda_driven_features:交互作用特徴量を生成中...


[2026-08-08 14:07:46] [INFO] 特徴量処理完了


INFO:12_ensemble_gbdt_eda_driven_features:特徴量処理完了


## 5. 特徴量の統合とEDA由来の派生特徴量

まず全特徴量セットを統合し、その後に「職種内偏差」「給与の相対化」「多項式」「ビン化」「複合交互作用」といった
**Train統計量ベースで計算する派生特徴量**を追加する（Trainのグループ平均・分位点をTestにも適用することでリークを防ぐ）。

In [13]:
logger.info("-" * 60)
logger.info("特徴量の統合")
logger.info("-" * 60)

train_persona_features = train_persona.drop(columns=[TARGET_COL])

# 月次集約特徴量を統合
train_features = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
train_features = train_features.merge(train_cat_change, on=ID_COL, how="left")
train_features = train_features.merge(train_missing, on=ID_COL, how="left")
train_features = train_features.merge(train_domain, on=ID_COL, how="left")
train_features = train_features.merge(train_advanced_stats, on=ID_COL, how="left")
train_features = train_features.merge(train_cluster, on=ID_COL, how="left")
train_features = train_features.merge(train_dept_te, on=ID_COL, how="left")
train_features = train_features.merge(train_eda_feats, on=ID_COL, how="left")
train_features = train_features.merge(train_mgr, on=ID_COL, how="left")

test_features = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
test_features = test_features.merge(test_cat_change, on=ID_COL, how="left")
test_features = test_features.merge(test_missing, on=ID_COL, how="left")
test_features = test_features.merge(test_domain, on=ID_COL, how="left")
test_features = test_features.merge(test_advanced_stats, on=ID_COL, how="left")
test_features = test_features.merge(test_cluster, on=ID_COL, how="left")
test_features = test_features.merge(test_dept_te, on=ID_COL, how="left")
test_features = test_features.merge(test_eda_feats, on=ID_COL, how="left")
test_features = test_features.merge(test_mgr, on=ID_COL, how="left")

logger.info(f"Train: {train_features.shape}, Test: {test_features.shape}")

[2026-08-08 14:07:46] [INFO] ------------------------------------------------------------


INFO:12_ensemble_gbdt_eda_driven_features:------------------------------------------------------------


[2026-08-08 14:07:46] [INFO] 特徴量の統合


INFO:12_ensemble_gbdt_eda_driven_features:特徴量の統合


[2026-08-08 14:07:46] [INFO] ------------------------------------------------------------


INFO:12_ensemble_gbdt_eda_driven_features:------------------------------------------------------------


[2026-08-08 14:07:46] [INFO] Train: (2761, 316), Test: (2502, 316)


INFO:12_ensemble_gbdt_eda_driven_features:Train: (2761, 316), Test: (2502, 316)


In [14]:
logger.info("[NEW] 職種別の乖離特徴量を生成中...")
# 初回レポート18.2「追加2」/ 「研修時間の負の相関は職種構造に起因する見かけの相関」という知見への対応
job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
job_means = {m: train_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
category_means_train = {m: train_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}

for df in [train_features, test_features]:
    for m in job_dev_metrics:
        job_mean_series = df["初期職種"].map(job_means[m])
        df[f"{m}_job_deviation"] = df[m] - job_mean_series
    # 研修時間の文脈化特徴量（初回レポート18.2「追加5」）
    df["研修時間_職種比"] = df["研修時間_mean"] / df["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
    df["研修時間_区分比"] = df["研修時間_mean"] / df["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)

logger.info("[NEW] 給与の相対化特徴量を生成中...")
# EDA v2レポート2節: 初任給・月例給与にTrain/Test分布シフトあり(KS統計量0.17〜0.20)。
# 絶対額に加え、等級内・入社区分内の相対値を追加し分布シフトへの頑健性を高める
grade_salary_mean = train_features.groupby("初期等級")["初任給_円"].mean().to_dict()
category_salary_mean = train_features.groupby("入社区分")["初任給_円"].mean().to_dict()
grade_monthly_salary_mean = train_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

for df in [train_features, test_features]:
    df["初任給_等級内偏差"] = df["初任給_円"] - df["初期等級"].map(grade_salary_mean)
    df["初任給_区分内偏差"] = df["初任給_円"] - df["入社区分"].map(category_salary_mean)
    df["月例給与_等級内偏差"] = df["月例給与_円_mean"] - df["初期等級"].map(grade_monthly_salary_mean)

logger.info("[NEW] 多項式特徴量を生成中...")
# 11_計画の未実装分: EDA v2で強い多重共線性が確認された年齢・前職経験・初任給、
# および月次データで最も重要度が高かった残業時間について2次項を追加
for df in [train_features, test_features]:
    df["入社時年齢_sq"] = df["入社時年齢"] ** 2
    df["前職経験月数_sq"] = df["前職経験月数"] ** 2
    df["初任給_円_sq"] = (df["初任給_円"] / 10000) ** 2  # スケール調整
    df["残業時間_mean_sq"] = df["残業時間_mean"] ** 2

logger.info("[NEW] ビン化特徴量を生成中...")
# 11_計画の未実装分。Trainの分位点で境界を決め、Testにもそのまま適用（両端は±infに拡張し外挿に対応）
salary_edges = pd.qcut(train_features["初任給_円"], q=5, retbins=True, duplicates="drop")[1]
salary_edges[0], salary_edges[-1] = -np.inf, np.inf
overtime_edges = pd.qcut(train_features["残業時間_mean"].dropna(), q=5, retbins=True, duplicates="drop")[1]
overtime_edges[0], overtime_edges[-1] = -np.inf, np.inf

for df in [train_features, test_features]:
    df["初任給_bin"] = pd.cut(df["初任給_円"], bins=salary_edges, labels=False)
    df["残業時間_mean_bin"] = pd.cut(df["残業時間_mean"], bins=overtime_edges, labels=False)

logger.info("[NEW] 複雑な交互作用特徴量(3変数)を生成中...")
# 11_計画の未実装分: 入社区分 × 初期職種 × 入社四半期(Q2かどうか)
for df in [train_features, test_features]:
    df["combo_job_category_q2"] = (
        df["combo_job_category"].astype(str) + "_" + np.where(df["入社四半期"] == 2, "Q2", "other")
    )

logger.info(f"派生特徴量生成後 Train: {train_features.shape}, Test: {test_features.shape}")

[2026-08-08 14:07:46] [INFO] [NEW] 職種別の乖離特徴量を生成中...


INFO:12_ensemble_gbdt_eda_driven_features:[NEW] 職種別の乖離特徴量を生成中...


[2026-08-08 14:07:46] [INFO] [NEW] 給与の相対化特徴量を生成中...


INFO:12_ensemble_gbdt_eda_driven_features:[NEW] 給与の相対化特徴量を生成中...


[2026-08-08 14:07:46] [INFO] [NEW] 多項式特徴量を生成中...


...

INFO:12_ensemble_gbdt_eda_driven_features:[NEW] ビン化特徴量を生成中...


[2026-08-08 14:07:46] [INFO] [NEW] 複雑な交互作用特徴量(3変数)を生成中...


INFO:12_ensemble_gbdt_eda_driven_features:[NEW] 複雑な交互作用特徴量(3変数)を生成中...


[2026-08-08 14:07:46] [INFO] 派生特徴量生成後 Train: (2761, 331), Test: (2502, 331)


INFO:12_ensemble_gbdt_eda_driven_features:派生特徴量生成後 Train: (2761, 331), Test: (2502, 331)


In [15]:
# カテゴリカル変数の処理
cat_cols = [
    "入社区分", "性別", "専攻分野", "採用経路", "初期職種", "初期勤務地", "初期役割",
    "combo_job_category", "combo_job_category_q2",  # [NEW] 複雑な交互作用カテゴリ
]
for col in cat_cols:
    if col in train_features.columns:
        le = LabelEncoder()
        combined = pd.concat([
            train_features[col].fillna("missing").astype(str),
            test_features[col].fillna("missing").astype(str),
        ])
        le.fit(combined)
        train_features[col] = le.transform(train_features[col].fillna("missing").astype(str))
        test_features[col] = le.transform(test_features[col].fillna("missing").astype(str))

# 不要な列の削除
# 「初期部署ID」は Target Encoding (dept_target_enc) に情報を抽出済みのため生IDは除外
# 「最終学歴」はEDAで統計的に非有意(カイ二乗検定 p=0.221)と確認済みのため除外
drop_cols = [ID_COL, "入社日", "入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
             "初期部署ID", "初期等級", "最終学歴", "前職職種"]
drop_cols_exist = [col for col in drop_cols if col in train_features.columns]
train_features = train_features.drop(columns=drop_cols_exist)
test_features = test_features.drop(columns=drop_cols_exist)

# NaNを-999で埋める
X_train = train_features.fillna(-999)
y_train_target = y_train.copy()
X_test = test_features.fillna(-999)

logger.info(f"X_train Shape: {X_train.shape}, X_test Shape: {X_test.shape}")
logger.info(f"最終特徴量数: {X_train.shape[1]}")

import optuna

[2026-08-08 14:07:46] [INFO] X_train Shape: (2761, 322), X_test Shape: (2502, 322)


INFO:12_ensemble_gbdt_eda_driven_features:X_train Shape: (2761, 322), X_test Shape: (2502, 322)


[2026-08-08 14:07:46] [INFO] 最終特徴量数: 322


INFO:12_ensemble_gbdt_eda_driven_features:最終特徴量数: 322


## 6. モデリング（CatBoost / LightGBM / XGBoost）

11_と同じ2段階学習の枠組み（初期パラメータで学習 → 重要度0の特徴量を除外 → Optunaでチューニング → 深い再学習）を維持しつつ、
以下を変更する：

- **TimeSeriesSplitの分割数を5→6に増加**: 11_ではOOF(0.585)とPublic(0.607)の差が大きく、fold数の少なさによるOOFの楽観性が疑われるため。
- **XGBoostのOptuna試行回数を10→20に増加**: 11_でXGBoostが3モデル中最も弱く(OOF 0.611)、Stackingでもほぼ無視されていたため探索を強化。
- **CatBoost/LightGBMのOptuna試行回数を10→15に増加**: 特徴量数が増えた（275→300強）ことに対応。

In [16]:
N_SPLITS = 6  # 11_の5から増加（OOFの安定性向上のため）

logger.info("=" * 60)
logger.info("CatBoostモデルの学習 (1回目)")
logger.info("=" * 60)

# sort_colとして入社日を使用
train_features_with_date = train_persona[[ID_COL, '入社日']].merge(
    pd.DataFrame({ID_COL: train_ids}), on=ID_COL, how='right'
)
X_train_with_sort = X_train.copy()
X_train_with_sort['入社日'] = train_features_with_date['入社日'].values

input_data_cat = {
    "X_train": X_train_with_sort,
    "y_train": y_train_target,
    "X_test": X_test,
    "sort_col": "入社日",
}

cat_params = {
    "n_splits": N_SPLITS,
    "seed": SEED,
    "save_dir": str(SAVED_MODELS_DIR / "catboost"),
    "cv_strategy": "timeseries",
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "l2_leaf_reg": 3,
    "border_count": 128,
    "bagging_temperature": 0.2,
    "random_strength": 1,
    "early_stopping_rounds": 50,
    "verbose": False,
    "task_type": "GPU",  # GPU対応
}

logger.info("CatBoostトレーニング開始 (1回目)...")
result_cat, _ = run_cat(input_data_cat, cat_params)

# ==========================================
# 特徴量重要度を計測し、不要な特徴量を削除
# ==========================================
if "models" in result_cat and len(result_cat["models"]) > 0:
    logger.info("CatBoostの特徴量重要度を計測し、重要度0の特徴量を削除します")
    cat_models = result_cat["models"]

    feature_names = cat_models[0].feature_names_
    feature_importance_sum = np.zeros(len(feature_names))

    for model in cat_models:
        feature_importance_sum += model.get_feature_importance()

    feature_importance_avg = feature_importance_sum / len(cat_models)

    # 重要度が0より大きい特徴量のみ保持
    important_features = [feat for feat, imp in zip(feature_names, feature_importance_avg) if imp > 0]

    logger.info(f"全{len(feature_names)}特徴量中、重要度が0の{len(feature_names) - len(important_features)}個を削除")

    # データセットの更新
    input_data_cat["X_train"] = X_train_with_sort[important_features + ["入社日"]]
    input_data_cat["X_test"] = X_test[important_features]

    # ==========================================
    # Optunaによるハイパーパラメータ探索 (GPU)
    # ==========================================
    logger.info("CatBoostのOptunaモジュールによる探索開始...")

    cat_params["n_trials"] = 15  # 11_の10から増加
    _, best_params = run_cat_optuna(input_data_cat, cat_params)
    logger.info(f"Best CatBoost Params: {best_params}")

    # ベストパラメータで深い再学習
    logger.info("CatBoostベストパラメータでの深い再学習開始...")
    cat_params.update(best_params)
    cat_params["iterations"] = 2000
    cat_params["early_stopping_rounds"] = 100
    result_cat, _ = run_cat(input_data_cat, cat_params)

cat_oof_score = result_cat["oof_score"]
cat_test_preds = result_cat["test_preds"]
cat_oof_preds = result_cat["oof_preds"]
logger.info(f"CatBoost OOF Score (LogLoss): {cat_oof_score:.6f}")

[2026-08-08 14:07:47] [INFO] ============================================================


INFO:12_ensemble_gbdt_eda_driven_features:============================================================


[2026-08-08 14:07:47] [INFO] CatBoostモデルの学習 (1回目)


INFO:12_ensemble_gbdt_eda_driven_features:CatBoostモデルの学習 (1回目)


[2026-08-08 14:07:47] [INFO] ============================================================


...

INFO:12_ensemble_gbdt_eda_driven_features:Best CatBoost Params: {'n_splits': 6, 'seed': 42, 'save_dir': '/content/drive/MyDrive/jaggle_2026/saved_models/20260808/12_ensemble_gbdt_eda_driven_features/catboost', 'cv_strategy': 'timeseries', 'iterations': 1000, 'learning_rate': 0.026480798913884752, 'depth': 3, 'l2_leaf_reg': 0.48382291633924523, 'border_count': 128, 'bagging_temperature': 0.7242393339184411, 'random_strength': 0.09844310537357427, 'early_stopping_rounds': 50, 'verbose': False, 'task_type': 'GPU', 'n_trials': 15}


[2026-08-08 14:27:16] [INFO] CatBoostベストパラメータでの深い再学習開始...


INFO:12_ensemble_gbdt_eda_driven_features:CatBoostベストパラメータでの深い再学習開始...


[2026-08-08 14:27:52] [INFO] CatBoost OOF Score (LogLoss): 0.589052


INFO:12_ensemble_gbdt_eda_driven_features:CatBoost OOF Score (LogLoss): 0.589052


In [17]:
logger.info("=" * 60)
logger.info("LightGBMモデルの学習 (1回目)")
logger.info("=" * 60)

input_data_lgb = {
    "X_train": X_train_with_sort,
    "y_train": y_train_target,
    "X_test": X_test,
    "sort_col": "入社日",
}

lgb_params = {
    "n_splits": N_SPLITS,
    "seed": SEED,
    "save_dir": str(SAVED_MODELS_DIR / "lightgbm"),
    "objective": "binary",
    "metric": "binary_logloss",
    "early_stopping_rounds": 50,
    "verbose_eval": False,
    "cv_strategy": "timeseries",
    "boosting_type": "gbdt",
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "min_child_samples": 20,
    "verbose": -1,
    "n_estimators": 1000,
    "device": "gpu",  # GPU対応
}

logger.info("LightGBMトレーニング開始 (1回目)...")
result_lgb, _ = run_lgb(input_data_lgb, lgb_params)

# ==========================================
# 特徴量重要度を計測し、不要な特徴量を削除
# ==========================================
if "models" in result_lgb and len(result_lgb["models"]) > 0:
    logger.info("LightGBMの特徴量重要度を計測し、重要度0の特徴量を削除します")
    lgb_models = result_lgb["models"]

    feature_names = lgb_models[0].feature_name()
    feature_importance_sum = np.zeros(len(feature_names))

    for model in lgb_models:
        feature_importance_sum += model.feature_importance(importance_type='gain')

    feature_importance_avg = feature_importance_sum / len(lgb_models)

    important_features = [feat for feat, imp in zip(feature_names, feature_importance_avg) if imp > 0]

    logger.info(f"全{len(feature_names)}特徴量中、重要度が0の{len(feature_names) - len(important_features)}個を削除")

    input_data_lgb["X_train"] = X_train_with_sort[important_features + ["入社日"]]
    input_data_lgb["X_test"] = X_test[important_features]

    logger.info("LightGBMのOptunaモジュールによる探索開始...")

    lgb_params["n_trials"] = 15  # 11_の10から増加
    _, best_params = run_lgb_optuna(input_data_lgb, lgb_params)
    logger.info(f"Best LightGBM Params: {best_params}")

    logger.info("LightGBMベストパラメータでの深い再学習開始...")
    lgb_params.update(best_params)
    lgb_params["n_estimators"] = 2000
    lgb_params["early_stopping_rounds"] = 100
    result_lgb, _ = run_lgb(input_data_lgb, lgb_params)

lgb_oof_score = result_lgb["oof_score"]
lgb_test_preds = result_lgb["test_preds"]
lgb_oof_preds = result_lgb["oof_preds"]
logger.info(f"LightGBM OOF Score (LogLoss): {lgb_oof_score:.6f}")

[2026-08-08 14:27:53] [INFO] ============================================================


INFO:12_ensemble_gbdt_eda_driven_features:============================================================


[2026-08-08 14:27:53] [INFO] LightGBMモデルの学習 (1回目)


INFO:12_ensemble_gbdt_eda_driven_features:LightGBMモデルの学習 (1回目)


[2026-08-08 14:27:53] [INFO] ============================================================


...

INFO:12_ensemble_gbdt_eda_driven_features:Best LightGBM Params: {'n_splits': 6, 'seed': 42, 'save_dir': '/content/drive/MyDrive/jaggle_2026/saved_models/20260808/12_ensemble_gbdt_eda_driven_features/lightgbm', 'objective': 'binary', 'metric': 'binary_logloss', 'early_stopping_rounds': 50, 'verbose_eval': False, 'cv_strategy': 'timeseries', 'boosting_type': 'gbdt', 'num_leaves': 41, 'learning_rate': 0.10459854480992457, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'min_child_samples': 47, 'verbose': -1, 'n_estimators': 1000, 'device': 'gpu', 'n_trials': 15, 'max_depth': 5, 'subsample': 0.7503933039619332, 'colsample_bytree': 0.6769474312859594, 'reg_alpha': 0.0005986257553604068, 'reg_lambda': 2.641559365216617e-08}


[2026-08-08 14:29:01] [INFO] LightGBMベストパラメータでの深い再学習開始...


INFO:12_ensemble_gbdt_eda_driven_features:LightGBMベストパラメータでの深い再学習開始...


[2026-08-08 14:29:05] [INFO] LightGBM OOF Score (LogLoss): 0.590282


INFO:12_ensemble_gbdt_eda_driven_features:LightGBM OOF Score (LogLoss): 0.590282


In [18]:
logger.info("=" * 60)
logger.info("XGBoostモデルの学習 (1回目)")
logger.info("=" * 60)

input_data_xgb = {
    "X_train": X_train_with_sort,
    "y_train": y_train_target,
    "X_test": X_test,
    "sort_col": "入社日",
}

xgb_params = {
    "n_splits": N_SPLITS,
    "seed": SEED,
    "save_dir": str(SAVED_MODELS_DIR / "xgboost"),
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "early_stopping_rounds": 50,
    "cv_strategy": "timeseries",
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1,
    "n_estimators": 1000,
    "tree_method": "hist",
    "device": "cuda",  # XGBoost 2.0以降のGPU対応
    "verbose": False,
}

logger.info("XGBoostトレーニング開始 (1回目)...")
result_xgb, _ = run_xgb(input_data_xgb, xgb_params)

# ==========================================
# 特徴量重要度を計測し、不要な特徴量を削除
# ==========================================
if "models" in result_xgb and len(result_xgb["models"]) > 0:
    logger.info("XGBoostの特徴量重要度を計測し、重要度0の特徴量を削除します")
    xgb_models = result_xgb["models"]

    feature_names = xgb_models[0].feature_names
    feat_imp_dict = {feat: 0.0 for feat in feature_names}

    for model in xgb_models:
        score = model.get_score(importance_type='gain')
        for k, v in score.items():
            feat_imp_dict[k] += v

    important_features = [feat for feat, imp in feat_imp_dict.items() if imp > 0]

    logger.info(f"全{len(feature_names)}特徴量中、重要度が0の{len(feature_names) - len(important_features)}個を削除")

    input_data_xgb["X_train"] = X_train_with_sort[important_features + ["入社日"]]
    input_data_xgb["X_test"] = X_test[important_features]

    logger.info("XGBoostのOptunaモジュールによる探索開始...")

    xgb_params["n_trials"] = 20  # 11_の10から増加（3モデル中最も弱かったため重点的に強化）
    _, best_params = run_xgb_optuna(input_data_xgb, xgb_params)
    logger.info(f"Best XGBoost Params: {best_params}")

    logger.info("XGBoostベストパラメータでの深い再学習開始...")
    xgb_params.update(best_params)
    xgb_params["n_estimators"] = 2000
    xgb_params["early_stopping_rounds"] = 100
    result_xgb, _ = run_xgb(input_data_xgb, xgb_params)

xgb_oof_score = result_xgb["oof_score"]
xgb_test_preds = result_xgb["test_preds"]
xgb_oof_preds = result_xgb["oof_preds"]
logger.info(f"XGBoost OOF Score (LogLoss): {xgb_oof_score:.6f}")

[2026-08-08 14:29:05] [INFO] ============================================================


INFO:12_ensemble_gbdt_eda_driven_features:============================================================


[2026-08-08 14:29:05] [INFO] XGBoostモデルの学習 (1回目)


INFO:12_ensemble_gbdt_eda_driven_features:XGBoostモデルの学習 (1回目)


[2026-08-08 14:29:05] [INFO] ============================================================


...

INFO:12_ensemble_gbdt_eda_driven_features:Best XGBoost Params: {'n_splits': 6, 'seed': 42, 'save_dir': '/content/drive/MyDrive/jaggle_2026/saved_models/20260808/12_ensemble_gbdt_eda_driven_features/xgboost', 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'early_stopping_rounds': 50, 'cv_strategy': 'timeseries', 'max_depth': 8, 'learning_rate': 0.19792585061233833, 'subsample': 0.9993746097718079, 'colsample_bytree': 0.8160963325792664, 'min_child_weight': 20, 'n_estimators': 1000, 'tree_method': 'hist', 'device': 'cuda', 'verbose': False, 'n_trials': 20, 'alpha': 0.00046071514713534015, 'lambda': 6.033831118608275e-06}


[2026-08-08 14:29:25] [INFO] XGBoostベストパラメータでの深い再学習開始...


INFO:12_ensemble_gbdt_eda_driven_features:XGBoostベストパラメータでの深い再学習開始...


[2026-08-08 14:29:26] [INFO] XGBoost OOF Score (LogLoss): 0.603422


INFO:12_ensemble_gbdt_eda_driven_features:XGBoost OOF Score (LogLoss): 0.603422


## 7. アンサンブル

11_の3手法（Simple Average / Weighted Average / Stacking）に加えて、以下を新規追加する：

- **Optimized Weighted Average**: 11_の重み付け平均は「OOFスコアの逆数」というヒューリスティックだった。
  ここではOOF Log Lossを直接最小化する重みを`scipy.optimize.minimize`で求める（重みの和=1、非負制約）。
- **Stacking（正則化強度をCVで選択）**: EDA v2レポート4節で年齢・前職経験・初任給に強い多重共線性（VIF 7〜8台）が確認されたため、
  固定の`LogisticRegression(C=1.0)`ではなく`LogisticRegressionCV`でL2正則化強度を内部CVで選択する。

In [19]:
logger.info("=" * 60)
logger.info("アンサンブル")
logger.info("=" * 60)

# モデル内部で '入社日' でソートされているため、ターゲットも同様にソートする
sort_idx = X_train_with_sort.sort_values('入社日').index
y_train_sorted = y_train_target.iloc[sort_idx].reset_index(drop=True)

# TimeSeriesSplitの仕様上、最初の学習データ部分にはOOF予測が存在せず0.0となるため、
# 有効な予測が行われたインデックスのみを抽出する
valid_idx = cat_oof_preds > 0
y_valid = y_train_sorted[valid_idx]

# 1. Simple Average
logger.info("1. Simple Average")
simple_avg_test = (cat_test_preds + lgb_test_preds + xgb_test_preds) / 3
simple_avg_oof = (cat_oof_preds + lgb_oof_preds + xgb_oof_preds) / 3
simple_avg_oof_score = calculate_logloss(y_valid, simple_avg_oof[valid_idx])
logger.info(f"Simple Average OOF Score: {simple_avg_oof_score:.6f}")

# 2. Weighted Average (スコアの逆数を重みとする、11_と同じヒューリスティック)
logger.info("2. Weighted Average (heuristic: 1/score)")
scores = np.array([cat_oof_score, lgb_oof_score, xgb_oof_score])
heuristic_weights = (1 / scores) / (1 / scores).sum()
logger.info(f"Weights: CatBoost={heuristic_weights[0]:.4f}, LightGBM={heuristic_weights[1]:.4f}, XGBoost={heuristic_weights[2]:.4f}")

weighted_avg_test = (
    cat_test_preds * heuristic_weights[0]
    + lgb_test_preds * heuristic_weights[1]
    + xgb_test_preds * heuristic_weights[2]
)
weighted_avg_oof = (
    cat_oof_preds * heuristic_weights[0]
    + lgb_oof_preds * heuristic_weights[1]
    + xgb_oof_preds * heuristic_weights[2]
)
weighted_avg_oof_score = calculate_logloss(y_valid, weighted_avg_oof[valid_idx])
logger.info(f"Weighted Average OOF Score: {weighted_avg_oof_score:.6f}")

# 3. [NEW] Optimized Weighted Average (OOF Log Lossを直接最小化)
logger.info("3. [NEW] Optimized Weighted Average")

def _neg_ll_for_weights(w):
    w_norm = np.abs(w) / np.sum(np.abs(w))
    blend = (
        w_norm[0] * cat_oof_preds[valid_idx]
        + w_norm[1] * lgb_oof_preds[valid_idx]
        + w_norm[2] * xgb_oof_preds[valid_idx]
    )
    return calculate_logloss(y_valid, blend)

opt_result = minimize(_neg_ll_for_weights, x0=[1 / 3, 1 / 3, 1 / 3], method="Nelder-Mead")
opt_weights = np.abs(opt_result.x) / np.sum(np.abs(opt_result.x))
logger.info(f"Optimized Weights: CatBoost={opt_weights[0]:.4f}, LightGBM={opt_weights[1]:.4f}, XGBoost={opt_weights[2]:.4f}")

opt_weighted_test = (
    cat_test_preds * opt_weights[0] + lgb_test_preds * opt_weights[1] + xgb_test_preds * opt_weights[2]
)
opt_weighted_oof = (
    cat_oof_preds * opt_weights[0] + lgb_oof_preds * opt_weights[1] + xgb_oof_preds * opt_weights[2]
)
opt_weighted_oof_score = calculate_logloss(y_valid, opt_weighted_oof[valid_idx])
logger.info(f"Optimized Weighted Average OOF Score: {opt_weighted_oof_score:.6f}")

# 4. Stacking (L2正則化強度をCVで選択)
logger.info("4. Stacking with LogisticRegressionCV (regularization tuned via CV)")
oof_stack = np.column_stack([cat_oof_preds, lgb_oof_preds, xgb_oof_preds])
test_stack = np.column_stack([cat_test_preds, lgb_test_preds, xgb_test_preds])

# EDA v2レポート4節: 年齢・前職経験・初任給に強い多重共線性(VIF 7〜8台)が確認されており、
# メタモデルの正則化強度は固定値ではなくCVで選択する
meta_model = LogisticRegressionCV(
    Cs=[0.01, 0.1, 1.0, 10.0, 100.0],
    cv=5,
    scoring="neg_log_loss",
    max_iter=1000,
    random_state=SEED,
)
meta_model.fit(oof_stack[valid_idx], y_valid)
stacking_test = meta_model.predict_proba(test_stack)[:, 1]
stacking_oof_valid = meta_model.predict_proba(oof_stack[valid_idx])[:, 1]

stacking_oof = np.zeros_like(cat_oof_preds)
stacking_oof[valid_idx] = stacking_oof_valid

stacking_oof_score = calculate_logloss(y_valid, stacking_oof_valid)
logger.info(f"Stacking OOF Score: {stacking_oof_score:.6f}")
logger.info(f"Selected C: {meta_model.C_[0]:.4f}")
logger.info(f"Meta model coefs: CatBoost={meta_model.coef_[0][0]:.4f}, LightGBM={meta_model.coef_[0][1]:.4f}, XGBoost={meta_model.coef_[0][2]:.4f}")

# スコアサマリ
logger.info("=" * 60)
logger.info("スコアサマリ")
logger.info("=" * 60)
logger.info(f"CatBoost OOF Score:              {cat_oof_score:.6f}")
logger.info(f"LightGBM OOF Score:              {lgb_oof_score:.6f}")
logger.info(f"XGBoost OOF Score:               {xgb_oof_score:.6f}")
logger.info(f"Simple Average OOF Score:        {simple_avg_oof_score:.6f}")
logger.info(f"Weighted Average OOF Score:      {weighted_avg_oof_score:.6f}")
logger.info(f"Optimized Weighted OOF Score:    {opt_weighted_oof_score:.6f}")
logger.info(f"Stacking OOF Score:              {stacking_oof_score:.6f}")
logger.info("=" * 60)

[2026-08-08 14:29:26] [INFO] ============================================================


INFO:12_ensemble_gbdt_eda_driven_features:============================================================


[2026-08-08 14:29:26] [INFO] アンサンブル


INFO:12_ensemble_gbdt_eda_driven_features:アンサンブル


[2026-08-08 14:29:26] [INFO] ============================================================


...

INFO:12_ensemble_gbdt_eda_driven_features:Optimized Weighted OOF Score:    0.584866


[2026-08-08 14:29:27] [INFO] Stacking OOF Score:              0.584923


INFO:12_ensemble_gbdt_eda_driven_features:Stacking OOF Score:              0.584923


[2026-08-08 14:29:27] [INFO] ============================================================


INFO:12_ensemble_gbdt_eda_driven_features:============================================================


## 8. 結果の保存

In [20]:
logger.info("-" * 60)
logger.info("結果の保存")
logger.info("-" * 60)

# Simple Average
sub_simple = pd.DataFrame({ID_COL: test_ids, TARGET_COL: simple_avg_test})
sub_simple.to_csv(SUBMISSION_SIMPLE_PATH, index=False, header=False)
logger.info(f"Simple Average保存: {SUBMISSION_SIMPLE_PATH}")

# Weighted Average
sub_weighted = pd.DataFrame({ID_COL: test_ids, TARGET_COL: weighted_avg_test})
sub_weighted.to_csv(SUBMISSION_WEIGHTED_PATH, index=False, header=False)
logger.info(f"Weighted Average保存: {SUBMISSION_WEIGHTED_PATH}")

# [NEW] Optimized Weighted Average
sub_opt_weighted = pd.DataFrame({ID_COL: test_ids, TARGET_COL: opt_weighted_test})
sub_opt_weighted.to_csv(SUBMISSION_OPT_WEIGHTED_PATH, index=False, header=False)
logger.info(f"Optimized Weighted Average保存: {SUBMISSION_OPT_WEIGHTED_PATH}")

# Stacking
sub_stacking = pd.DataFrame({ID_COL: test_ids, TARGET_COL: stacking_test})
sub_stacking.to_csv(SUBMISSION_STACKING_PATH, index=False, header=False)
logger.info(f"Stacking保存: {SUBMISSION_STACKING_PATH}")

logger.info("=" * 60)
logger.info("=== 実験完了 ===")
logger.info("=" * 60)

print(f"\n■ 個別モデルOOF Scores:")
print(f"  CatBoost:  {cat_oof_score:.6f}")
print(f"  LightGBM:  {lgb_oof_score:.6f}")
print(f"  XGBoost:   {xgb_oof_score:.6f}")
print(f"\n■ アンサンブルOOF Scores:")
print(f"  Simple Avg:          {simple_avg_oof_score:.6f}")
print(f"  Weighted Avg:        {weighted_avg_oof_score:.6f}")
print(f"  Optimized Weighted:  {opt_weighted_oof_score:.6f}")
print(f"  Stacking:            {stacking_oof_score:.6f}")
print(f"\n■ 保存ファイル:")
print(f"  Simple Average:      {SUBMISSION_SIMPLE_PATH}")
print(f"  Weighted Average:    {SUBMISSION_WEIGHTED_PATH}")
print(f"  Optimized Weighted:  {SUBMISSION_OPT_WEIGHTED_PATH}")
print(f"  Stacking:            {SUBMISSION_STACKING_PATH}")

[2026-08-08 14:29:27] [INFO] ------------------------------------------------------------


INFO:12_ensemble_gbdt_eda_driven_features:------------------------------------------------------------


[2026-08-08 14:29:27] [INFO] 結果の保存


INFO:12_ensemble_gbdt_eda_driven_features:結果の保存


[2026-08-08 14:29:27] [INFO] ------------------------------------------------------------


...

[2026-08-08 14:29:27] [INFO] === 実験完了 ===


INFO:12_ensemble_gbdt_eda_driven_features:=== 実験完了 ===


[2026-08-08 14:29:27] [INFO] ============================================================


INFO:12_ensemble_gbdt_eda_driven_features:============================================================



■ 個別モデルOOF Scores:
  CatBoost:  0.589052
  LightGBM:  0.590282
  XGBoost:   0.603422

■ アンサンブルOOF Scores:
  Simple Avg:          0.587813
  Weighted Avg:        0.587744
  Optimized Weighted:  0.584866
  Stacking:            0.584923

■ 保存ファイル:
  Simple Average:      /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_12_ensemble_gbdt_eda_driven_features_simple_avg.csv
  Weighted Average:    /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_12_ensemble_gbdt_eda_driven_features_weighted_avg.csv
  Optimized Weighted:  /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_12_ensemble_gbdt_eda_driven_features_opt_weighted_avg.csv
  Stacking:            /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_12_ensemble_gbdt_eda_driven_features_stacking.csv


## 9. まとめ（今回の改善点と次のアクション）

### 今回実装した改善点
1. 部署IDのTarget Encoding（KFold + スムージング）を新規追加 — 11_で完全に未活用だった最有力候補への対応
2. 欠勤日数パターン・360度評価タイミング・上司チーム規模など、EDAレポートで指摘された知見を直接特徴量化
3. 11_で計画のみで未実装だった多項式・ビン化・複雑な交互作用・時系列ラグ・比率特徴量を実装
4. 給与関連特徴量をTrain/Test分布シフトに頑健な相対値（等級内偏差・区分内偏差）で補強
5. TimeSeriesSplitの分割数増加、XGBoostのOptuna試行増加、Optimized Weighted Average、
   正則化強度をCV選択するStackingにより、モデリング面も改善

### 次のアクション候補（本ノートブックでも未対応）
- 早期離職（12-23ヶ月）と長期離職（60ヶ月以降）を区別したアプローチ（EDA v2レポート7.2節: 60ヶ月以降の離職者は
  0-23ヶ月のデータでは定着者とほぼ見分けがつかないという定量的な限界が判明している）
- テキスト特徴量のTF-IDF/埋め込みベース特徴量（初回レポート8.3節、文字数以外は未実装）
- カテゴリカル変数のOne-Hot Encoding（現状は全てLabelEncoding。CatBoostのネイティブカテゴリ処理に一部依存）
